In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_clientes = 60000

# Safra de observação (foto dos dados). O default, quando ocorre, é 12 meses depois.
# 4 anos: 75% acumulados de 2022 a 2024 para treino, 25% em 2025 para teste
safra_obs = np.random.choice(
    [2022, 2023, 2024, 2025],
    size=n_clientes,
    p=[0.20, 0.25, 0.30, 0.25]
)

# Dados cadastrais e do cartão na safra de observação
renda_mensal = np.random.lognormal(mean=8.2, sigma=0.5, size=n_clientes).round(2)
limite = (renda_mensal * np.random.uniform(1.5, 4.0, size=n_clientes)).round(2)

# Taxa de utilização (alguns clientes já estão com 100% do limite tomado)
utilizacao_inicial = np.clip(np.random.beta(a=2, b=3, size=n_clientes) * 1.15, 0.0, 1.0)
saldo = (limite * utilizacao_inicial).round(2)

# Sinais comportamentais
variacao_saldo_3m = np.random.normal(loc=0.15, scale=0.25, size=n_clientes).round(4)
qtd_atrasos_curtos_6m = np.random.poisson(lam=0.8, size=n_clientes)
pagou_minimo_ult_fatura = np.random.binomial(n=1, p=0.35, size=n_clientes)

# Target PD: default em 12 meses (> 90 dias de atraso)
logit_pd = (
    -4.5
    + 0.55 * qtd_atrasos_curtos_6m
    + 1.6 * utilizacao_inicial
    + 1.5 * np.clip(variacao_saldo_3m, -0.5, 1.0)
    + 0.35 * pagou_minimo_ult_fatura
    + np.random.normal(0, 0.6, size=n_clientes)
)
prob_default = 1 / (1 + np.exp(-logit_pd))
default_12m = np.random.binomial(n=1, p=prob_default)

# Comportamento até o calote (usado no modelo de EAD)
score_estresse = (
    0.30 * utilizacao_inicial
    + 0.25 * np.clip(variacao_saldo_3m, -0.5, 1.0)
    + 0.12 * qtd_atrasos_curtos_6m
    + 0.20 * pagou_minimo_ult_fatura
    + np.random.normal(0, 0.12, size=n_clientes)
)
ccf_verdadeiro = np.clip(score_estresse, 0.0, 1.0)

# Saldo registrado no dia do default: só existe para quem deu default
limite_livre = limite - saldo
saldo_default = (
    saldo + ccf_verdadeiro * limite_livre + np.random.normal(0, 50, size=n_clientes)
).round(2)
saldo_default = np.maximum(saldo_default, 0.0)
saldo_default = np.where(default_12m == 1, saldo_default, np.nan)

df = pd.DataFrame({
    "safra_obs": safra_obs,
    "renda_mensal": renda_mensal,
    "limite": limite,
    "saldo": saldo,
    "utilizacao_inicial": utilizacao_inicial,
    "variacao_saldo_3m": variacao_saldo_3m,
    "qtd_atrasos_curtos_6m": qtd_atrasos_curtos_6m,
    "pagou_minimo_ult_fatura": pagou_minimo_ult_fatura,
    "default_12m": default_12m,
    "saldo_default": saldo_default,
})

df.to_csv('Dados.csv', index=False)